In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import os

In [3]:
# os.environ["WANDB_DISABLED"] = "true"

In [4]:
df = pd.read_csv('../data/train.csv')
df.head()

,id,text,anger,fear,joy,sadness,surprise,emotions
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness']
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness']
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness']
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness']
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear']


In [5]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
for col in emotion_cols:
    df[col] = df[col].astype(int)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
labels = emotion_cols
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

In [8]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

In [9]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
X_train = train_df['text'].tolist()
y_train = train_df[labels].values.tolist()
X_val = val_df['text'].tolist()
y_val = val_df[labels].values.tolist()

In [10]:
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    # Apply sigmoid and threshold
    sigmoid_preds = torch.sigmoid(torch.Tensor(preds))
    binary_preds = (sigmoid_preds > 0.5).int().numpy()
    
    true_labels = p.label_ids

    f1_macro = f1_score(y_true=true_labels, y_pred=binary_preds, average='macro', zero_division=0)
    
    return {
        'f1': f1_macro
    }

# Distil BERT

In [11]:
MODEL_CHECKPOINT = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [12]:
train_dataset = EmotionDataset(X_train, y_train, tokenizer)
val_dataset = EmotionDataset(X_val, y_val, tokenizer)

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [14]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\91762\AppData\Local\Temp\ipykernel_27784\4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1
1,0.427300,0.377935,0.638569
2,0.316600,0.329515,0.729674
3,0.262000,0.313530,0.739842


TrainOutput(global_step=1026, training_loss=0.3544779028344108, metrics={'train_runtime': 433.1599, 'train_samples_per_second': 37.822, 'train_steps_per_second': 2.369, 'total_flos': 542582375051520.0, 'train_loss': 0.3544779028344108, 'epoch': 3.0})

## For Test Dataset

In [16]:
df_test = pd.read_csv('../data/test.csv')
test_texts = df_test['text'].tolist()

In [17]:
num_labels = len(emotion_cols)
dummy_labels = [[0] * num_labels for _ in range(len(test_texts))]

In [18]:
test_dataset = EmotionDataset(texts=test_texts, labels=dummy_labels, tokenizer=tokenizer)
raw_predictions = trainer.predict(test_dataset)

In [19]:
test_logits = raw_predictions.predictions
sigmoid_preds = torch.sigmoid(torch.Tensor(test_logits))
binary_preds = (sigmoid_preds > 0.5).int().numpy()
print(binary_preds[:5])

[[1 1 0 1 0]
 [0 0 0 0 0]
 [1 1 0 0 1]
 [0 1 0 0 0]
 [0 1 0 0 1]]


In [42]:
df_submission = pd.DataFrame(binary_preds, columns=emotion_cols)
df_submission.insert(0, 'id', df_test['id'])
# df_submission.to_csv('submission.csv', index=False)
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,1,0
1,1,0,0,0,0,0
2,2,1,1,0,0,1
3,3,0,1,0,0,0
4,4,0,1,0,0,1


# RoBERTa

In [5]:
MODEL_CHECKPOINT = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Projects\DL-and-Gen-AI-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91762\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
train_dataset = EmotionDataset(X_train, y_train, tokenizer)
val_dataset = EmotionDataset(X_val, y_val, tokenizer)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.to(device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [12]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\91762\AppData\Local\Temp\ipykernel_15496\4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.398400,0.338113,0.722663
2,0.291200,0.299151,0.765872
3,0.231800,0.272471,0.784066


TrainOutput(global_step=1026, training_loss=0.32655106139229517, metrics={'train_runtime': 871.4514, 'train_samples_per_second': 18.8, 'train_steps_per_second': 1.177, 'total_flos': 1077666131996928.0, 'train_loss': 0.32655106139229517, 'epoch': 3.0})

## For Test Dataset

In [15]:
df_test = pd.read_csv('../data/test.csv')
test_texts = df_test['text'].tolist()

In [16]:
num_labels = len(emotion_cols)
dummy_labels = [[0] * num_labels for _ in range(len(test_texts))]

In [17]:
test_dataset = EmotionDataset(texts=test_texts, labels=dummy_labels, tokenizer=tokenizer)
raw_predictions = trainer.predict(test_dataset)

In [18]:
test_logits = raw_predictions.predictions
sigmoid_preds = torch.sigmoid(torch.Tensor(test_logits))
binary_preds = (sigmoid_preds > 0.5).int().numpy()
print(binary_preds[:5])

[[1 1 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 1]
 [0 1 0 0 0]
 [0 1 0 0 1]]


In [20]:
df_submission = pd.DataFrame(binary_preds, columns=emotion_cols)
df_submission.insert(0, 'id', df_test['id'])
# df_submission.to_csv('submission.csv', index=False)
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,0,0
1,1,0,0,0,0,0
2,2,1,1,0,0,1
3,3,0,1,0,0,0
4,4,0,1,0,0,1


# BERT

In [24]:
MODEL_CHECKPOINT = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [25]:
train_dataset = EmotionDataset(X_train, y_train, tokenizer)
val_dataset = EmotionDataset(X_val, y_val, tokenizer)

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [28]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\91762\AppData\Local\Temp\ipykernel_3924\4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.400000,0.345017,0.702240
2,0.274800,0.296653,0.777119
3,0.208800,0.277338,0.787653


TrainOutput(global_step=1026, training_loss=0.318420480798792, metrics={'train_runtime': 927.7861, 'train_samples_per_second': 17.658, 'train_steps_per_second': 1.106, 'total_flos': 1077666131996928.0, 'train_loss': 0.318420480798792, 'epoch': 3.0})

## For Test Dataset

In [26]:
df_test = pd.read_csv('../data/test.csv')
test_texts = df_test['text'].tolist()

In [27]:
num_labels = len(emotion_cols)
dummy_labels = [[0] * num_labels for _ in range(len(test_texts))]

In [28]:
test_dataset = EmotionDataset(texts=test_texts, labels=dummy_labels, tokenizer=tokenizer)
raw_predictions = trainer.predict(test_dataset)

In [29]:
test_logits = raw_predictions.predictions
sigmoid_preds = torch.sigmoid(torch.Tensor(test_logits))
binary_preds = (sigmoid_preds > 0.5).int().numpy()
print(binary_preds[:5])

[[1 1 0 1 0]
 [0 0 0 0 0]
 [1 1 0 0 1]
 [0 1 0 0 0]
 [0 1 0 0 1]]


In [30]:
df_submission = pd.DataFrame(binary_preds, columns=emotion_cols)
df_submission.insert(0, 'id', df_test['id'])
# df_submission.to_csv('submission.csv', index=False)
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,1,0
1,1,0,0,0,0,0
2,2,1,1,0,0,1
3,3,0,1,0,0,0
4,4,0,1,0,0,1


# MILESTONE 3

In [13]:
classifier_dropout_specific = model.config.classifier_dropout
print(f"The 'classifier_dropout' parameter is: {classifier_dropout_specific}")

classifier_dropout_fallback = model.config.hidden_dropout_prob
print(f"The fallback 'hidden_dropout_prob' parameter is: {classifier_dropout_fallback}")

The 'classifier_dropout' parameter is: None
The fallback 'hidden_dropout_prob' parameter is: 0.1


In [2]:
MODEL_CHECKPOINT = 'bert-base-uncased'
BATCH_SIZE = 32
SEQUENCE_LENGTH = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Testing with batch size: {BATCH_SIZE} on device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

dummy_texts = ["This is a test sentence."] * BATCH_SIZE
dummy_labels = torch.randint(0, 2, (BATCH_SIZE, 5)).float().to(device)

inputs = tokenizer(dummy_texts, padding='max_length', max_length=SEQUENCE_LENGTH, truncation=True, return_tensors="pt").to(device)

try:
    outputs = model(**inputs, labels=dummy_labels)
    loss = outputs.loss
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    print(f"SUCCESS: Batch size {BATCH_SIZE} fits in memory.")
    
except torch.cuda.OutOfMemoryError:
    print(f"FAILED: Batch size {BATCH_SIZE} is too large and caused an Out of Memory error.")

torch.cuda.empty_cache()

Testing with batch size: 32 on device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SUCCESS: Batch size 32 fits in memory.
